# Partition Problem
(see also [Wikipedia](https://en.wikipedia.org/wiki/Partition_problem))

The partition problem for a set of numbers $\mathcal{S} = \{a_1, ..., a_N\}$ consists of finding two subsets $\mathcal{S}_{1} \cup \mathcal{S}_2 = \mathcal{S}$ such that the difference of the sums over the two subsets $\mathcal{S}_{1, 2}$ is as small as possible. Spin $\sigma_i = \pm 1$ says which subset $a_i$ joins, and the QAOA minimizes the squared difference
$$
C = \left(\sum_{i=1}^{N} a_i \sigma_i\right)^2 = \sum_{i=1}^{N} a_i^2 + \sum_{i<j\leq N} J_{ij} \sigma_i \sigma_j
$$
with $J_{ij} = 2a_i a_j$.

In [ ]:
using QAOA, LinearAlgebra, Random

using PyPlot
PyPlot.plt.style.use("paper.mplstyle");

__Defining the problem by hand:__

In [ ]:
N = 4
a = rand(Xoshiro(1), N)

In [ ]:
J = 2 .* (a * transpose(a))
J[diagind(J)] .= 0.0
J

In [ ]:
partition = IsingProblem(zeros(N), J; offset = sum(abs2, a))

__Using the wrapper function:__

In [ ]:
partition = partition_problem(a)

__Optimizing the angles:__ `optimize_schedule` tunes the angles of `p` layers and returns them with the average squared difference they give:

In [ ]:
p = 4
angles, value = optimize_schedule(partition, p)

In [ ]:
probs = probabilities(partition, angles)

xlabels = []
for bstr in digits.(0:2^N-1, base=2, pad=N)
    push!(xlabels, "\$|" * prod([string(b) for b in bstr]) * "\\rangle\$")
end

figure(figsize=(5, 3.2))
ax = subplot(111)
bar(0:2^N-1, probs)
ax.set_xticks(0:2^N-1)
ax.set_xticklabels(xlabels, rotation=90)
minorticks_off()
tight_layout()

`solve` measures the final state and returns the best split it saw:

In [ ]:
result = solve(partition, angles)
a[result.σ .== 1], a[result.σ .== -1]